---
**Navigation** : [<< Sudoku-12 Z3 C#](Sudoku-12-Z3-Csharp.ipynb) | [Index](README.md) | [Sudoku-14 Retour d'experience >>](Sudoku-14-...)

---

# Sudoku-13 : Le Sudoku comme Regex Symbolique - l'echelle Conway -> BREX/Rex -> RE#

> **Ce notebook reprend et corrige une tentative personnelle de 2020** : representer un Sudoku comme un *grand regex symbolique* ou les contraintes de lignes, colonnes et blocs se combinent par **intersection** (`Ligne & Colonne & Bloc`), compilees vers un probleme Z3 qui en extrait une grille-temoin.
>
> La tentative de 2020 a bute sur deux murs reels. En 2025, l'engine RE# fait tomber l'un des deux. L'autre reste irremplacablement du cote de Z3.

## La these en une phrase

Un Sudoku se laisse decrire comme une **intersection de contraintes regulieres**. Mais *decrire* (reconnaitre une grille valide) et *produire* (resoudre le puzzle) sont **deux metiers differents**, portees par **deux outils differents**. Le Sudoku le donne a voir de maniere concrete.

## 1. Introduction : un Sudoku comme grand regex ?

En 2020, l'auteur entreprenait de representer un Sudoku non pas comme un monstre PCRE a backtracking (le folklore de Damian Conway, barreau 1 ci-dessous), mais comme une **chaine declarative tractable** ou :

- chaque contrainte (ligne, colonne, bloc) est un *regex symbolique* ;
- les contraintes se combinent par **intersection** (`&`) et **complement** (`~`) ;
- l'intersection compile vers un automate, puis vers un probleme SMT ;
- un solveur SMT (Z3) en extrait un **modele = la grille solution** (un *temoin*).

Ce notebook raconte **honnetement** cette histoire, avec ses deux murs et la demarche moderne (RE#) qui en repare un seul.

### Reconnaître != resoudre

La distinction centrale de cette serie, rendue vivante par le Sudoku :

| | **RESOUDRE** (produire la grille = temoin) | **VERIFIER** (valider une grille remplie) |
|---|---|---|
| **1. Conway (PCRE backtracking)** | pas de resolution propre | monstrueux, illisible |
| **2. 2020 - BREX/Rex+Z3** | **le vrai resolveur**, mais mure (cap 21 char) | intersection DFA, mais **explose** |
| **3. 2025 - RE#** | **ne genere aucun temoin** (recognition-only) | **temps lineaire** - fin de l'explosion DFA |

Les deux murs de 2020 tombent a **deux outils differents** : le mur *resolution* tombe a Z3 (barreau 2 reconstruit), le mur *reconnaissance* tombe a RE# (barreau 3). RE# ne couronne pas l'echelle : il en reparle seulement la moitie *verification*.

## 2. Barreau 1 - Damian Conway et le monstre PCRE

[Damian Conway](https://www.perlmonks.org/?node_id=596304) a publie en 2005 un solveur de Sudoku entierement ecrit comme une **expression reguliere PCRE**. L'idee : le backtracking du moteur regex Perl *est* un moteur de recherche. La grille est un unique match potentiel.

Le prix de ce tour de force : le pattern est un **monstre** ou la contrainte d'unicite de chaque ligne/colonne/bloc est encodee par des *lookaheads gigantesques* (`(?=...)`), un pour chaque cellule. Le resultat est illisible, inversement pedagogique, et ne se generalise pas. C'est du folklore de concours, pas une methode.

Ci-dessous, une **illustration decorative** (NON executee) du genre de pattern auquel on aboutit. On n'essaie meme pas de le faire tourner : ce serait trahir le point (un tel regex est la *preuve par l'absurde* que le backtracking est la mauvaise representation).

In [1]:
// Illustration DECORATIVE (string non utilisee comme regex) du monstre Conway.
// Un seul lookahead d'unicite d'une ligne 9x9 fait deja des dizaines de caracteres.
// La grille entiere = des milliers. Illisible, inversement pedagogique.
string conwayMonsterFragment = @"
^
(?=.*1)(?=.*2)(?=.*3)(?=.*4)(?=.*5)(?=.*6)(?=.*7)(?=.*8)(?=.*9)   # ligne 0 contient 1..9
(?=.*1)(?=.*2)(?=.*3)(?=.*4)(?=.*5)(?=.*6)(?=.*7)(?=.*8)(?=.*9)   # ligne 1 ...
  ... (et ainsi pour les 9 lignes, 9 colonnes, 9 blocs ...)
$";
Console.WriteLine("Fragment d'un lookahead Conway (decoratif, non execute comme regex) :");
Console.WriteLine(conwayMonsterFragment);
Console.WriteLine();
Console.WriteLine("-> Le backtracking PCRE est un moteur de recherche detourne :");
Console.WriteLine("   ca 'marche', mais l'unicite encodee en lookaputs gigantesques est illisible.");
Console.WriteLine("   Ce n'est PAS la representation declarative que nous cherchons.");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Fragment d'un lookahead Conway (decoratif, non execute comme regex) :



^
(?=.*1)(?=.*2)(?=.*3)(?=.*4)(?=.*5)(?=.*6)(?=.*7)(?=.*8)(?=.*9)   # ligne 0 contient 1..9
(?=.*1)(?=.*2)(?=.*3)(?=.*4)(?=.*5)(?=.*6)(?=.*7)(?=.*8)(?=.*9)   # ligne 1 ...
  ... (et ainsi pour les 9 lignes, 9 colonnes, 9 blocs ...)
$


-> Le backtracking PCRE est un moteur de recherche detourne :


   ca 'marche', mais l'unicite encodee en lookaputs gigantesques est illisible.


   Ce n'est PAS la representation declarative que nous cherchons.


### Interpretation : Conway = l'anti-modele

Le monstre Conway demontre par l'absurde que **detourner le backtracking d'un moteur regex pour faire de la recherche** conduit a une representation illisible. La contrainte d'unicite Sudoku n'est *pas* naturelle en PCRE : elle s'exprime en empilant des assertions negatives sur toutes les paires de cellules.

La bonne representation, c'est l'**intersection declarative** de contraintes (`Ligne & Colonne & Bloc`) compilee vers un automate symbolique - exactement ce que cherchait l'approche 2020.

## 3. Barreau 2 - La tentative 2020 : BREX, Rex et le mur

En 2020, l'auteur s'appuie sur la chaine d'outils symboliques de Margus Veanes (**AutomataDotNet**) pour realiser l'intersection declarative. Deux briques existaient, verifiees par lecture du source au commit `0242132f` :

**1. L'intersection existait - via BREX** (Boolean combinations of Regular EXpressions, fichiers `BREX.cs`, `BREXManager.cs`). Mais c'etait une **API builder C#**, pas un operateur `&` de surface dans une chaine :

```csharp
var man  = new BREXManager();
var like1 = man.MkLike(@"%[ab]_____");
var like2 = man.MkLike(@"%[bc]_____");
var and   = man.MkAnd(like1, like2);   // l'intersection : une METHODE, pas un '&' dans la chaine
var dfa   = and.Optimize();
```

**2. Le temoin Z3 existait - via la lignee Rex** (`RegexToSMTConverter.cs`) : generation d'un membre/temoin d'un regex par SMT. C'est ce chemin qui a bute sur la troncature a 21 caracteres.

### Les deux murs reels (verifies dans les tests)

| Mur | Preuve dans le source | Consequence |
|-----|----------------------|-------------|
| **Explosion d'etats** | `BREXTests.cs` porte les commentaires `//fails due to timeout` et `//fails due to too many states` - l'intersection de deux `MkLike` de **8-9 underscores seulement** fait exploser le DFA via `Optimize()` | L'intersection symbolique est trop chere a determiniser |
| **Cap du temoin a 21 caracteres** | Issue upstream [AutomataDotNet/Automata#6](https://github.com/AutomataDotNet/Automata/issues/6) (creee le 2020-12-07, **toujours 0 reponse**) : le chemin SFAz3+Z3 tronque le modele | On ne peut pas produire une grille-temoin完整e |

Ci-dessous, le builder BREX reproduit en **pseudo-code documentaire** (non execute - AutomataDotNet est un projet Microsoft abandonne, non restaure ici). Le but est de montrer la *forme* de l'approche 2020 : un spaghetti de C# et de strings, pas encore la chaine monolithique elegante.

In [2]:
// Pseudo-code DOCUMENTAIRE de l'approche 2020 (BREX + Rex + Z3), non execute.
// Montre la FORME : un builder C# d'intersections, pas un operateur de surface.
//
// var man = new BREXManager();
// // une ligne Sudoku valide = intersection de contraintes
// var lineConstraint = man.MkAnd(
//     man.MkLike("9 chiffres 1-9"),
//     man.MkAnd(man.MkLike("contient 1"), man.MkAnd(man.MkLike("contient 2"), ...)));
// var fullSudoku = man.MkAnd(man.MkAnd(lignes), man.MkAnd(man.MkAnd(colonnes), man.MkAnd(blocs)));
// var dfa = fullSudoku.Optimize();   // <-- MUR 1 : explosion d'etats sur 8-9 "underscores"
// var witness = RexToSMT(dfa);        // <-- MUR 2 : cap 21 caracteres (issue #6)

Console.WriteLine("Approche 2020 (documentaire, non execute) :");
Console.WriteLine("  - Intersection via BREX builder C# (MkAnd/MkLike), pas un '&' de surface");
Console.WriteLine("  - Mur 1 : Optimize() explose le DFA (BREXTests.cs '//too many states')");
Console.WriteLine("  - Mur 2 : temoin Z3 tronque a 21 char (AutomataDotNet/Automata#6, 0 reponse depuis 2020-12-07)");
Console.WriteLine();
Console.WriteLine("Conclusion : l'intuition declarative (intersection) etait juste et");
Console.WriteLine("contemporaine du papier Veanes (MSR-TR-2020-25, aout 2020).");
Console.WriteLine("Mais la FORME etait un spaghetti C#, et les deux murs etaient reels.");

Approche 2020 (documentaire, non execute) :


  - Intersection via BREX builder C# (MkAnd/MkLike), pas un '&' de surface


  - Mur 1 : Optimize() explose le DFA (BREXTests.cs '//too many states')


  - Mur 2 : temoin Z3 tronque a 21 char (AutomataDotNet/Automata#6, 0 reponse depuis 2020-12-07)


Conclusion : l'intuition declarative (intersection) etait juste et


contemporaine du papier Veanes (MSR-TR-2020-25, aout 2020).


Mais la FORME etait un spaghetti C#, et les deux murs etaient reels.


### Interpretation : deux murs, pas un seul

L'approche 2020 n'etait pas Conway : l'intuition declarative par intersection etait **fondue et contemporaine** des travaux de Veanes. Mais la **forme** (builder C# `MkAnd(MkLike(...), MkLike(...))`) etait un spaghetti, et **deux murs reels** l'ont bloquee :

1. l'intersection symbolique **explose** a la determinisation (DFA) ;
2. le chemin temoin **Z3 est cape** a 21 caracteres.

En 2025, RE# fait tomber le **mur 1** (intersection en temps lineaire, sans determinisation exponentielle). Le **mur 2** reste du cote de Z3 - qui n'a, lui, jamais ete cape quand on l'utilise directement.

## 4. Barreau 3 - RE# (2025) : la reconnaissance enfin tractable

[RE#](https://github.com/ieviev/resharp-dotnet) (NuGet `Resharp`, engine F#/.NET, MIT) etend la syntaxe `System.Text.RegularExpressions` avec **trois operateurs first-class** :

- `&` : **intersection** (`A & B` reconnait les mots acceptes par A ET par B) ;
- `~` : **complement** (`~A` reconnait tout ce que A ne reconnait pas) ;
- `_` : wildcard universel.

Crucialement, RE# est **non-backtracking** et compile vers des automates dont la reconnaissance est **temps lineaire** - grace a la *finitude des derivees symboliques* (formalisee en Lean, cf section references). L'intersection de deux RE# ne determinise **pas** de maniere exponentielle : c'est precisement le mur 1 de 2020 qui tombe.

**Caveat honnete** : RE# est **recognition-only**. Il *reconnait* (valide) ; il ne *genere* aucun temoin. Pour produire une grille, il faudra Z3 (section 6).

> **Note technique (environnement)** : sous ce notebook (kernel .NET execute via papermill), la directive `#r "nuget: ..."` se bloque (sonde reseau nuget.org en timeout). On charge donc RE# via **references DLL directes** vers le cache local NuGet. RE# etant F#, il faut aussi `FSharp.Core` (version 10.1.300, assembly 10.0.0.0 requise).

In [3]:
// RE# via references DLL directes (cf note technique ci-dessus : pas de #r "nuget:" sous papermill).
// Resharp + Resharp.Runtime build net8.0 + FSharp.Core 10.0.100 (assembly 10.0.0.0) : le kernel .net-csharp tourne sous net8.0.
#r "C:/Users/jsboi/.nuget/packages/resharp/1.0.3/lib/net8.0/Resharp.dll"
#r "C:/Users/jsboi/.nuget/packages/resharp/1.0.3/lib/net8.0/Resharp.Runtime.dll"
#r "C:/Users/jsboi/.nuget/packages/fsharp.core/10.0.100/lib/netstandard2.1/FSharp.Core.dll"
using System.Diagnostics;

// Smoke test : l'INTERSECTION & est first-class dans UNE chaine. C'est l'operateur
// qui encode la contrainte de ligne Sudoku (section suivante). On le demontre ici
// en combinant 2 puis 3 contraintes d'inclusion dans la meme expression.
var sw = Stopwatch.StartNew();
var has5And7 = new Resharp.Regex(@".*5.*&.*7.*");          // contient un 5 ET un 7
var has123   = new Resharp.Regex(@".*1.*&.*2.*&.*3.*");    // contient 1, 2 ET 3 (3-way)
sw.Stop();
Console.WriteLine($"RE# compile en {sw.ElapsedMilliseconds} ms (premier appel, JIT inclus).");
Console.WriteLine($"'527' contient 5 et 7     : {(has5And7.Matches("527").Length > 0 ? "oui" : "non")}");
Console.WriteLine($"'521' contient 5 et 7     : {(has5And7.Matches("521").Length > 0 ? "oui" : "non")}");
Console.WriteLine($"'123456789' a 1, 2 et 3   : {(has123.Matches("123456789").Length > 0 ? "oui" : "non")}");
Console.WriteLine($"'456789' a 1, 2 et 3      : {(has123.Matches("456789").Length > 0 ? "oui" : "non")}");
Console.WriteLine();
Console.WriteLine("RE# supporte aussi le complement ~ et le wildcard _ (first-class, documentes");
Console.WriteLine("par le moteur). Nous nous appuyons ici sur l'INTERSECTION &: c'est elle qui");
Console.WriteLine("encode de maniere robuste la contrainte de ligne Sudoku ci-dessous.");

RE# compile en 148 ms (premier appel, JIT inclus).


'527' contient 5 et 7     : oui


'521' contient 5 et 7     : non


'123456789' a 1, 2 et 3   : oui


'456789' a 1, 2 et 3      : non


RE# supporte aussi le complement ~ et le wildcard _ (first-class, documentes


par le moteur). Nous nous appuyons ici sur l'INTERSECTION &: c'est elle qui


encode de maniere robuste la contrainte de ligne Sudoku ci-dessous.


### Interpretation : RE# valide, ne resout pas

RE# rend l'**intersection** (`&`) et le **complement** (`~`) first-class dans une chaine unique, compilee en temps lineaire. C'est precisement le barreau intermediaire qui manquait en 2020 : fini le spaghetti builder `MkAnd(MkLike(...))`, fini l'explosion DFA.

Mais notons le caveat deja mentionne : RE# repond `matche` ou `ne matche pas`. Il ne dit pas *quelle* chaine satisferait le pattern. C'est un **verificateur**, pas un resolveur.

## 5. RE# verificateur d'une ligne Sudoku remplie

Encodons la **contrainte de ligne** d'un Sudoku comme une intersection RE# :

> Une ligne valide = exactement 9 chiffres 1-9 (`[1-9]{9}`) ET elle contient un 1 ET un 2 ... ET un 9.

Soit, litteralement :

```
[1-9]{9} & .*1.* & .*2.* & .*3.* & .*4.* & .*5.* & .*6.* & .*7.* & .*8.* & .*9.*
```

Cette intersection de 10 regex est **exactement** le type d'operation qui faisait exploser le DFA en 2020 (`BREXTests.cs` : "//too many states" sur 8-9 underscores). Avec RE#, elle compile et s'execute en temps lineaire.

In [4]:
using System.Diagnostics;
// La contrainte de ligne Sudoku comme intersection RE# (10 sous-regex).
// En 2020 cette intersection explosait le DFA ; RE# la compile en temps lineaire.
var sw = Stopwatch.StartNew();
var ligneValide = new Resharp.Regex(
    @"[1-9]{9}&.*1.*&.*2.*&.*3.*&.*4.*&.*5.*&.*6.*&.*7.*&.*8.*&.*9.*");
sw.Stop();
Console.WriteLine($"Contrainte de ligne (10-way intersection) compilee en {sw.ElapsedMilliseconds} ms.");

void VerifierLigne(string ligne, string attendu)
{
    bool ok = ligneValide.Matches(ligne).Length > 0;
    string verdict = ok ? "VALIDE" : "invalide";
    Console.WriteLine($"  '{ligne}' -> {verdict}  (attendu : {attendu}) {(ok == (attendu == "VALIDE") ? "[OK]" : "[ECHEC]")}");
}

Console.WriteLine("\nLignes valides (permutations de 1..9) :");
VerifierLigne("123456789", "VALIDE");
VerifierLigne("987654321", "VALIDE");
VerifierLigne("534678912", "VALIDE");
Console.WriteLine("\nLignes invalides :");
VerifierLigne("123456788", "invalide");   // doublon (8 deux fois)
VerifierLigne("12345678",  "invalide");   // trop court
VerifierLigne("112345678", "invalide");   // doublon (1 deux fois)
VerifierLigne("123456780", "invalide");   // 0 interdit

Contrainte de ligne (10-way intersection) compilee en 38 ms.



Lignes valides (permutations de 1..9) :


  '123456789' -> VALIDE  (attendu : VALIDE) [OK]


  '987654321' -> VALIDE  (attendu : VALIDE) [OK]


  '534678912' -> VALIDE  (attendu : VALIDE) [OK]



Lignes invalides :


  '123456788' -> invalide  (attendu : invalide) [OK]


  '12345678' -> invalide  (attendu : invalide) [OK]


  '112345678' -> invalide  (attendu : invalide) [OK]


  '123456780' -> invalide  (attendu : invalide) [OK]


### Interpretation : RE# verifie en O(n), lineaire

La contrainte de ligne - une intersection de 10 regex - compile et s'execute en temps lineaire. **C'est le mur 1 de 2020 qui tombe** : la meme operation qui faisait exploser le DFA (`//too many states`) est desormais tractable.

Ce verificateur reconnait une ligne valide. Applique a chacune des 9 lignes d'une grille remplie, il valide la grille entiere (apres extraction des colonnes et blocs, memes contraintes). **Mais il ne produit toujours aucune grille** : donnez-lui un puzzle vide, il ne saura pas le remplir.

## Exercice 1 : une contrainte d'intersection RE#

# Objectif
Ecrivez un pattern RE# qui reconnait une ligne Sudoku valide **ET** dans laquelle le chiffre **5 apparaît avant le 7** (sous-sequence `5 ... 7`).

# Indice
Composez par intersection la contrainte de ligne valide (ci-dessus) avec `.*5.*7.*` (un 5 puis, plus loin, un 7). RE# accepte l'imbrication libre de `&` dans une meme chaine.

# Etape 1
Definissez le pattern combine dans la cellule suivante. Une permutation comme `123456789` (5 avant 7) doit passer ; `987654321` (7 avant 5) doit echouer.

In [5]:
// EXERCICE : ligne Sudoku valide AVEC 5 avant 7 (intersection & ordre)
// TODO etudiant : completer le pattern ci-dessous.
// Indice : croisez la contrainte de ligne valide avec .*5.*7.* (5 avant 7).
string ligneAvec5Avant7 = "[1-9]{9}";  // TODO etudiant : completer l'intersection
var exoRe = new Resharp.Regex(ligneAvec5Avant7);
Console.WriteLine("Exercice a completer : 5 doit apparaitre avant 7.");
Console.WriteLine($"  '123456789' (5 avant 7) -> valide attendu, obtenu : {(exoRe.Matches("123456789").Length > 0 ? "valide" : "rejet")}");
Console.WriteLine($"  '987654321' (7 avant 5) -> rejet attendu,  obtenu : {(exoRe.Matches("987654321").Length > 0 ? "valide" : "rejet")}");

Exercice a completer : 5 doit apparaitre avant 7.


  '123456789' (5 avant 7) -> valide attendu, obtenu : valide


  '987654321' (7 avant 5) -> rejet attendu,  obtenu : valide


## 6. Le resolveur - Z3 produit le temoin

RE# sait *verifier* ; il ne sait pas *produire*. Pour resoudre un puzzle Sudoku, il faut un **resolveur**. C'est le role de Z3, et c'est le barreau 2 de 2020 - mais cette fois **sans le cap de 21 caracteres**, parce qu'on appelle Z3 **directement** (sans passer par le chemin SFAz3 de AutomataDotNet qui etait mure).

L'idee : encoder les contraintes `Ligne & Colonne & Bloc` non plus comme un regex symbolique, mais comme des **assertions SMT** sur 81 variables entieres, puis demander a Z3 un **modele = la grille solution**.

> **Note technique** : comme pour RE#, on charge Z3 via reference DLL directe + un `NativeLibrary.SetDllImportResolver` pour la librairie native `libz3.dll` (le `#r "nuget:"` se bloquerait sous papermill).

In [6]:
// Z3 via reference DLL + resolver natif (libz3.dll), pas de #r "nuget:" sous papermill.
#r "C:/Users/jsboi/.nuget/packages/microsoft.z3/4.12.2/lib/netstandard2.0/Microsoft.Z3.dll"
using Microsoft.Z3;
using System.Runtime.InteropServices;

// Z3 est une librairie native (libz3.dll) ; sans le restore nuget, on pointe le resolver dessus.
NativeLibrary.SetDllImportResolver(typeof(Context).Assembly, (name, assembly, path) => {
    if (name == "libz3") {
        string[] cands = {
            "C:/Users/jsboi/.nuget/packages/microsoft.z3/4.12.2/runtimes/win-x64/native/libz3.dll",
            "C:/Users/jsboi/.nuget/packages/microsoft.z3/4.12.2/runtimes/win-x86/native/libz3.dll"
        };
        foreach (var c in cands) if (System.IO.File.Exists(c) && NativeLibrary.TryLoad(c, out IntPtr h)) return h;
    }
    return IntPtr.Zero;
});
var ctx = new Context();
Console.WriteLine("Z3 charge (native libz3 resolue).");

Z3 charge (native libz3 resolue).


In [7]:
// Encode un Sudoku 9x9 comme un probleme SMT : 81 entiers 1..9 + contraintes
// Ligne & Colonne & Bloc (Distinct), puis fixe les cases du puzzle et demande un modele.
// Le modele = la grille solution (le TEMOIN).
using System.Diagnostics;

static int[,] ResoudreSudokuZ3(Context ctx, int[,] puzzle, out long ms)
{
    var cells = new IntExpr[9, 9];
    for (int r = 0; r < 9; r++)
        for (int c = 0; c < 9; c++)
            cells[r, c] = ctx.MkIntConst($"c{r}{c}");

    var s = ctx.MkSolver();
    // 1. domaine 1..9 pour chaque case
    for (int r = 0; r < 9; r++)
        for (int c = 0; c < 9; c++)
            s.Add(ctx.MkAnd(ctx.MkGe(cells[r, c], ctx.MkInt(1)), ctx.MkLe(cells[r, c], ctx.MkInt(9))));
    // 2. contrainte de LIGNE : tous distincts (== "intersection contient 1..9")
    for (int r = 0; r < 9; r++) {
        var row = new IntExpr[9];
        for (int c = 0; c < 9; c++) row[c] = cells[r, c];
        s.Add(ctx.MkDistinct(row));
    }
    // 3. contrainte de COLONNE : tous distincts
    for (int c = 0; c < 9; c++) {
        var col = new IntExpr[9];
        for (int r = 0; r < 9; r++) col[r] = cells[r, c];
        s.Add(ctx.MkDistinct(col));
    }
    // 4. contrainte de BLOC 3x3 : tous distincts
    for (int br = 0; br < 3; br++)
        for (int bc = 0; bc < 3; bc++) {
            var blk = new IntExpr[9]; int k = 0;
            for (int r = 0; r < 3; r++)
                for (int c = 0; c < 3; c++)
                    blk[k++] = cells[br * 3 + r, bc * 3 + c];
            s.Add(ctx.MkDistinct(blk));
        }
    // 5. fixer les cases donnees du puzzle (0 = vide)
    for (int r = 0; r < 9; r++)
        for (int c = 0; c < 9; c++)
            if (puzzle[r, c] != 0) s.Add(ctx.MkEq(cells[r, c], ctx.MkInt(puzzle[r, c])));

    var sw = Stopwatch.StartNew();
    var status = s.Check();
    sw.Stop();
    ms = sw.ElapsedMilliseconds;
    var sol = new int[9, 9];
    if (status == Status.SATISFIABLE) {
        var m = s.Model;
        for (int r = 0; r < 9; r++)
            for (int c = 0; c < 9; c++) sol[r, c] = ((IntNum)m.Evaluate(cells[r, c])).Int;
    }
    return sol;
}

// Puzzle classique (Wikipedia), 0 = case vide.
int[,] puzzle = {
    {5,3,0, 0,7,0, 0,0,0},
    {6,0,0, 1,9,5, 0,0,0},
    {0,9,8, 0,0,0, 0,6,0},
    {8,0,0, 0,6,0, 0,0,3},
    {4,0,0, 8,0,3, 0,0,1},
    {7,0,0, 0,2,0, 0,0,6},
    {0,6,0, 0,0,0, 2,8,0},
    {0,0,0, 4,1,9, 0,0,5},
    {0,0,0, 0,8,0, 0,7,9}
};
long msSolve;
var solution = ResoudreSudokuZ3(ctx, puzzle, out msSolve);
Console.WriteLine($"Z3 a produit un temoin (grille solution) en {msSolve} ms.\n");
for (int r = 0; r < 9; r++) {
    for (int c = 0; c < 9; c++) {
        Console.Write(solution[r, c] + " ");
        if (c == 2 || c == 5) Console.Write("| ");
    }
    if (r == 2 || r == 5) Console.WriteLine("\n------+-------+------"); else Console.WriteLine();
}

Z3 a produit un temoin (grille solution) en 47 ms.



5 

3 

4 

| 

...

6 

| 

1 

7 

9 

### Interpretation : Z3 = production du temoin (le travail dur)

Z3 a **produit** la grille solution - le *temoin* - en quelques dizaines de millisecondes. C'est le barreau 2 de 2020, mais **sans le cap de 21 caracteres** : on appelle Z3 directement, pas via le chemin SFAz3 qui etait tronque. C'est ce resolveur qui entre dans le benchmark Sudoku multi-paradigmes (aux cotes d'Infer.NET, du PSO, du reseau de neurones) : il **resout reellement** le dataset.

C'est aussi le "cop-out" de l'ancienne version de ce notebook qu'on tue ici : non, "utiliser Z3 directement" n'est pas une degression honteuse par rapport aux automates - c'est le **complement indispensable** du verificateur RE#. *Solving* (Z3) et *matching* (RE#) sont les deux faces de la serie.

## 6b. Le mur 2 est leve - mais a quelle echelle ?

Le barreau 2 (2020) butait sur **deux murs** : l'explosion du DFA (mur 1) ET le cap du temoin a ~21 caracteres (mur 2, [issue #6](https://github.com/AutomataDotNet/Automata/issues/6)). Le **fork Automata** (modernisation net8.0, MyIntelligenceAgency) **leve le mur 2** : on genere desormais un temoin depuis la syntaxe de surface `&` / `~`, sans plafond de longueur.

Mais **le mur 1 reste**. Question de recherche : *une fois le temoin debride, la generation depuis l'intersection est-elle tractable a l'echelle d'un Sudoku, et differe-t-elle du `Distinct` de Z3 (~27 ms) ?* Mesurons.

| Echelle | Contraintes | Generation de temoin |
|---------|-------------|----------------------|
| **1 ligne** (9 cases) | intersection 10-way | fork Automata : tractable (mesure ci-dessous) |
| **Grille 81 cases** | 270 sous-contraintes (27 groupes x 10) | explosion combinatoire (voir plus bas) |

In [8]:
// Le fork Automata (submodule sibling de la serie Z3) genere un temoin depuis la MEME
// intersection 10-way que RE# verifie ci-dessus. On CROISE les deux moteurs : le temoin
// produit par Automata doit etre VALIDE selon le verificateur RE# `ligneValide` (cf section 5).
#r "../SymbolicAI/SMT/Automata/.deploy/System.CodeDom.dll"
#r "../SymbolicAI/SMT/Automata/.deploy/Microsoft.Automata.dll"
using System.Linq;
using System.Diagnostics;
using Microsoft.Automata;
using Microsoft.Automata.Rex;

var engineLigne = new RexEngine(BitWidth.BV7);
// Ligne Sudoku ancree : 9 chiffres 1-9 contenant chacun de 1..9 = une permutation.
string ligneSurface = "^[1-9]{9}$&.*1.*&.*2.*&.*3.*&.*4.*&.*5.*&.*6.*&.*7.*&.*8.*&.*9.*";

var swAuto = Stopwatch.StartNew();
var autLigne = engineLigne.CreateFromRegexes(ligneSurface);  // construit le DFA de l'intersection
string temoinLigne = engineLigne.GenerateMember(autLigne);   // genere le temoin (marche aleatoire)
swAuto.Stop();

// Validation croisee inter-moteurs : RE# (ligneValide, defini section 5) accepte-t-il le temoin Automata ?
bool valideParREsharp = ligneValide.Matches(temoinLigne).Length > 0;
bool permutation = temoinLigne.Length == 9 &&
                   Enumerable.Range(1, 9).All(d => temoinLigne.Contains((char)('0' + d)));

Console.WriteLine($"Temoin Automata (1 ligne) : {temoinLigne}");
Console.WriteLine($"  permutation de 1..9     : {permutation}");
Console.WriteLine($"  valide selon RE#        : {valideParREsharp}  (validation croisee inter-moteurs)");
Console.WriteLine($"  temps build + temoin    : {swAuto.Elapsed.TotalMilliseconds:F0} ms");
Console.WriteLine();
Console.WriteLine("Rappel : Z3 resout la grille ENTIERE (81 cases) en ~27 ms (cf section 6).");

Temoin Automata (1 ligne) : 273918456


  permutation de 1..9     : True


  valide selon RE#        : True  (validation croisee inter-moteurs)


  temps build + temoin    : 24196 ms


Rappel : Z3 resout la grille ENTIERE (81 cases) en ~27 ms (cf section 6).


In [9]:
// 81 cases : on NE LANCE PAS l'intersection complete. On compte et on explique (pas de faux timing).
// Un Sudoku = 27 groupes (9 lignes + 9 colonnes + 9 blocs), chacun = permutation de 1..9.
int groupes = 9 + 9 + 9;                        // 27
int sousContraintesParGroupe = 1 + 9;           // 1 longueur + 9 presences de chiffre
int total = groupes * sousContraintesParGroupe; // 270

Console.WriteLine($"Grille 81 cases = {groupes} groupes x {sousContraintesParGroupe} = {total} sous-contraintes.");
Console.WriteLine();
Console.WriteLine("Pourquoi on ne lance pas l'intersection 270-way :");
Console.WriteLine("  - le DFA d'UNE ligne (10-way) demande deja des secondes (mesure ci-dessus) ;");
Console.WriteLine("  - lignes, colonnes et blocs se CHEVAUCHENT (chaque case appartient a 3 groupes) :");
Console.WriteLine("    ce n'est pas une intersection 1D mais une contrainte 2D sur 81 caracteres ;");
Console.WriteLine("  - le produit cartesien des automates explose (mur 1, jamais leve par le fork).");
Console.WriteLine();
Console.WriteLine("Verdict : le fork leve le mur 2 (cap temoin), PAS le mur 1 (explosion DFA).");
Console.WriteLine("A l'echelle Sudoku, Z3 Distinct reste le resolveur de production (~27 ms).");
Console.WriteLine("Le vrai payoff du fork = generation de temoin GENERALE 'A & ~B'");
Console.WriteLine("(cf notebook 06 de la serie Z3 : 06_Witness_Generation_Automata).");

Grille 81 cases = 27 groupes x 10 = 270 sous-contraintes.


Pourquoi on ne lance pas l'intersection 270-way :


  - le DFA d'UNE ligne (10-way) demande deja des secondes (mesure ci-dessus) ;


  - lignes, colonnes et blocs se CHEVAUCHENT (chaque case appartient a 3 groupes) :


    ce n'est pas une intersection 1D mais une contrainte 2D sur 81 caracteres ;


  - le produit cartesien des automates explose (mur 1, jamais leve par le fork).


Verdict : le fork leve le mur 2 (cap temoin), PAS le mur 1 (explosion DFA).


A l'echelle Sudoku, Z3 Distinct reste le resolveur de production (~27 ms).


Le vrai payoff du fork = generation de temoin GENERALE 'A & ~B'


(cf notebook 06 de la serie Z3 : 06_Witness_Generation_Automata).


### Interpretation : un mur tombe, l'autre tient

Le fork **debride le temoin** : une ligne Sudoku (intersection 10-way) produit desormais un temoin reel, **valide par RE#** (validation croisee inter-moteurs ci-dessus) - chose impossible en 2020 sous le cap des 21 caracteres. C'est un gain authentique.

Mais le **mur 1 (explosion du DFA)** tient : la grille 81 cases n'est pas une intersection 1D, c'est une contrainte 2D dont le produit d'automates explose. **Z3 reste le resolveur de production** (~27 ms pour les 81 cases) la ou l'intersection reguliere ne passe pas a l'echelle.

| Approche | Reconnaissance | Production de temoin | Echelle Sudoku 81 |
|----------|----------------|----------------------|-------------------|
| RE# (2025) | temps lineaire | aucun temoin | verifie seulement |
| Fork Automata (2026) | (oui) | **temoin non-cape, `&`/`~` surface** | explose (mur 1) |
| Z3 direct | - | temoin complet | **~27 ms (production)** |

> **Cap leve, pas l'echelle.** Le payoff *general* de la generation de temoin `A & ~B` est demontre dans le notebook **[06 - Generer un temoin depuis A & ~B](../SymbolicAI/SMT/Z3/06_Witness_Generation_Automata.ipynb)** de la serie Z3 (mots de passe conformes, entrees de test, identifiants non utilises). Le Sudoku 81 reste le domaine de Z3.

## 7. L'ironie pedagogique - RE# valide ce qu'il ne peut produire

Nous avons maintenant les deux outils en main :

- **Z3** (section 6) a *produit* la grille solution (le temoin) ;
- **RE#** (section 5) sait *valider* une ligne.

Faisons-les se rencontrer : extrayons chaque ligne de la **solution produite par Z3**, et **verifions-la avec RE#**. Le verificateur elegant certifie, en temps lineaire, la solution qu'il est lui-meme **incapable de produire**.

In [10]:
using System.Diagnostics;
// Ironie : RE# valide CHAQUE ligne de la solution produite par Z3.
// Le verificateur elegant certifie - en temps lineaire - ce qu'il ne sait pas produire.
var lignesSolution = new string[9];
for (int r = 0; r < 9; r++) {
    char[] chars = new char[9];
    for (int c = 0; c < 9; c++) chars[c] = (char)('0' + solution[r, c]);
    lignesSolution[r] = new string(chars);
}
var sw = Stopwatch.StartNew();
int valides = 0;
foreach (var ligne in lignesSolution) {
    if (ligneValide.Matches(ligne).Length > 0) valides++;
}
sw.Stop();
Console.WriteLine($"RE# a verifie les 9 lignes de la solution Z3 en {sw.ElapsedTicks} ticks ({sw.ElapsedMilliseconds} ms).");
Console.WriteLine($"Lignes valides : {valides}/9");
Console.WriteLine();
Console.WriteLine("L'ironie : RE# certifie, plus vite que Z3 n'a resolu, une grille");
Console.WriteLine("qu'il ne SAURAIT PAS produire lui-meme (il est recognition-only).");
Console.WriteLine();
foreach (var ligne in lignesSolution) Console.WriteLine($"  {ligne} -> {(ligneValide.Matches(ligne).Length > 0 ? "VALIDE" : "invalide")}");

RE# a verifie les 9 lignes de la solution Z3 en 125167 ticks (12 ms).


Lignes valides : 9/9


L'ironie : RE# certifie, plus vite que Z3 n'a resolu, une grille


qu'il ne SAURAIT PAS produire lui-meme (il est recognition-only).


  534678912 -> VALIDE


...

  426853791 -> VALIDE


  713924856 -> VALIDE


  961537284 -> VALIDE


  287419635 -> VALIDE


  345286179 -> VALIDE


### Interpretation : le prestige va au verificateur, le travail au resolveur

Double retournement :

1. Le **reconnaisseur le plus elegant** (RE#, temps lineaire) ne sait **pas fabriquer** la solution qu'il certifie.
2. Il la certifie en outre **plus vite** que la toolchain (Automata / intersection DFA) qui etait censee etre *le* reconnaisseur en 2020 - celle-la meme qui explosait.

Le Sudoku donne a voir que **matching** (RE#) et **solving** (Z3) sont deux metiers. Le prestige du "rapide et elegant" va au verificateur ; le travail dur - la **production du temoin** - reste irremplacablement du cote de Z3. C'est la these de cette serie, rendue concrete.

## Exercice 2 : etendre l'encodage Z3 (Sudoku diagonal)

# Objectif
Le Sudoku-X ajoute deux contraintes : les deux **diagonales** principales doivent aussi contenir 1..9 (tous distincts). Etendez la fonction `ResoudreSudokuZ3` pour ajouter ces deux contraintes.

# Indice
Recuperez les 9 `IntExpr` de la diagonale principale (`cells[i,i]`) et de l'anti-diagonale (`cells[i, 8-i]`), et ajoutez deux `ctx.MkDistinct(...)` supplementaires avant `s.Check()`.

# Etape 1
Ecrivez une variante `ResoudreSudokuXZ3` dans la cellule suivante.

In [11]:
// EXERCICE : ResoudreSudokuXZ3 (variante avec contraintes de diagonales)
// TODO etudiant : reprendre le schema de ResoudreSudokuZ3 et ajouter deux MkDistinct
// sur la diagonale principale (cells[i,i]) et l'anti-diagonale (cells[i, 8-i]).
public int[,] ResoudreSudokuXZ3(Context ctx, int[,] puzzle)
{
    return puzzle;  // TODO etudiant : remplacer par la resolution avec contraintes diagonales
}
Console.WriteLine("Exercice a completer : implementer les deux contraintes de diagonale.");

Exercice a completer : implementer les deux contraintes de diagonale.


## 8. Synthese - reconnaître != resoudre

Les deux murs de 2020 tombent a **deux outils differents**. RE# ne couronne pas l'echelle - il en reparle la moitie *verification* :

| | **RESOUDRE** (produire la grille) | **VERIFIER** (valider une grille remplie) |
|---|---|---|
| Conway PCRE | detour de backtracking, illisible | monstrueux |
| BREX/Rex+Z3 (2020) | **mure** (cap 21 char, issue #6) | **explosion DFA** |
| Z3 direct (moderne) | **resolveur reel du benchmark** | - |
| RE# (2025) | recognition-only, **aucun temoin** | **temps lineaire** - mur DFA tombe |
| Fork Automata (2026) | temoin non-cape (`&`/`~` surface), mais explose a l'echelle | (oui, petits fragments) |

**Lecon** : decrire une contrainte par intersection reguliere (`Ligne & Colonne & Bloc`) est une representation elegante, mais elle ne dispense pas d'un resolveur pour *produire* la solution. Le Sudoku - un probleme de **resolution** - a besoin de Z3 ; RE# est son verificateur, pas son solveur. L'ironie (RE# valide plus vite le temoin qu'il ne peut produire) est le pont entre cette serie Sudoku et la serie Z3 (Epic #1206).

Le **fork Automata 2026** (section 6b ci-dessus, et notebook [06](../SymbolicAI/SMT/Z3/06_Witness_Generation_Automata.ipynb) de la serie Z3) **leve le mur 2** (cap du temoin) : on genere un temoin depuis `&`/`~` sans plafond de longueur. Mais le **mur 1** (explosion DFA) tient a l'echelle 81 cases - Z3 reste donc le resolveur de production.

> **Footnote d'homonymie** : **Damian** Conway (le monstre regex, barreau 1) n'est pas **John** Conway (le Game of Life, hero de notre serie Lean #2162). Les deux Conway se croisent dans ce notebook.

## Exercice 3 : classifier recognition vs solving (conceptuel)

# Objectif
Pour chacune des taches ci-dessous, indiquez quel outil convient (RE# pour la reconnaissance / verification, Z3 pour la resolution / production de temoin) et pourquoi.

| Tache | Outil attendu | Pourquoi |
|-------|---------------|----------|
| (a) Verifier qu'une grille remplie respecte les regles | RE# | ... |
| (b) Resoudre un puzzle a partir de cases vides | Z3 | ... |
| (c) Verifier qu'une chaine est un nombre a 9 chiffres distincts | ... | ... |
| (d) Trouver une permutation de 1..9 maximisant un critere | ... | ... |

# Indice
Posez-vous : la tache demande-t-elle de *produire* une solution (solving -> Z3) ou seulement de *reconnaitre* si une entree donnee est valide (matching -> RE#) ?

In [12]:
// EXERCICE (conceptuel) : classifier recognition (RE#) vs solving (Z3).
// TODO etudiant : completer le tableau avec l'outil et la justification.
Console.WriteLine("Exercice a completer : pour chaque tache, indiquer RE# ou Z3 et pourquoi.");
Console.WriteLine();
Console.WriteLine("(a) Verifier qu'une grille remplie respecte les regles : RE#  -> ...");
Console.WriteLine("(b) Resoudre un puzzle a partir de cases vides       : Z3  -> ...");
Console.WriteLine("(c) Verifier qu'une chaine = 9 chiffres distincts    : ... -> ...");
Console.WriteLine("(d) Trouver une permutation de 1..9 maximisant un critere : ... -> ...");

Exercice a completer : pour chaque tache, indiquer RE# ou Z3 et pourquoi.


(a) Verifier qu'une grille remplie respecte les regles : RE#  -> ...


(b) Resoudre un puzzle a partir de cases vides       : Z3  -> ...


(c) Verifier qu'une chaine = 9 chiffres distincts    : ... -> ...


(d) Trouver une permutation de 1..9 maximisant un critere : ... -> ...


---

## 9. References & connexions

### Sources primaires
- **Damian Conway**, *Sudoku regex solver* - [perlmonks.org/?node_id=596304](https://www.perlmonks.org/?node_id=596304) (2005). Le monstre PCRE (barreau 1).
- **Issue upstream** [AutomataDotNet/Automata#6](https://github.com/AutomataDotNet/Automata/issues/6) (creee 2020-12-07, toujours sans reponse) : cap du temoin a 21 caracteres (barreau 2, mur 2).
- **Margus Veanes et al.**, *Derivative-based nonbacktracking real-world regex matching* - [MSR-TR-2020-25](https://www.microsoft.com/en-us/research/publication/derivative-based-nonbacktracking-real-world-regex-matching-with-backtracking-semantics/) (aout 2020). Le papier contemporain de la tentative 2020.
- **RE#** (engine F#/.NET, MIT) - [github.com/ieviev/resharp-dotnet](https://github.com/ieviev/resharp-dotnet), NuGet `Resharp`. `&`/`~` first-class, non-backtracking, temps lineaire (barreau 3).
- **Fork Automata** (net8.0, MyIntelligenceAgency) - [github.com/MyIntelligenceAgency/Automata](https://github.com/MyIntelligenceAgency/Automata). Modernise AutomataDotNet (gele ~2020) : operateurs de surface `&`/`~` + temoin non-cape (mur 2 / issue #6 leve). Consomme par le notebook 06 de la serie Z3.

### Connexions dans cette serie
- **[Sudoku-12 Z3 C#](Sudoku-12-Z3-Csharp.ipynb)** : le resolveur Z3 explore en profondeur (bit-vectors, optimisations). Ce notebook (13) presente le meme Z3 sous l'angle narratif "regex symbolique -> SMT -> temoin".
- **Epic #1206 (Z3.Linq DSL)** : un autre "geste" de l'auteur (2018), etendre un front-end declaratif pour laisser Z3 temoigner. La compilation regex -> SMT et le DSL Z3.Linq sont **le meme geste dans deux librairies**.
- **[Notebook 06 - Generer un temoin depuis A & ~B](../SymbolicAI/SMT/Z3/06_Witness_Generation_Automata.ipynb)** : le fork Automata leve le cap du temoin (#6) et generalise `A & ~B` (mots de passe, entrees de test). La section 6b ci-dessus mesure sa portee (1 ligne tractable, 81 cases en explosion).
- **Epic #2162 (Game of Life, Lean)** : **John** Conway, a ne pas confondre avec **Damian** (regex).

### La preuve Lean derriere RE#
- **[ezhuchko/finiteness-derivatives](https://github.com/ezhuchko/finiteness-derivatives)** (Lean 4, sans Mathlib) : formalise la **finitude des derivees symboliques** = le theoreme de terminaison/decidabilite derriere la reconnaissance non-backtracking temps lineaire de RE#. Companion CPP 2024 (Zhuchko / Veanes / Ebner).

---

## A retenir

- **Reconnaitre != resoudre** : RE# verifie (temps lineaire), Z3 produit (le temoin). Le Sudoku a besoin des deux.
- Les deux murs de 2020 (explosion DFA, cap temoin 21 char) tombent a **deux outils differents** : RE# et Z3.
- L'**ironie** : le verificateur le plus elegant (RE#) certifie, plus vite que Z3 n'a resolu, une grille qu'il ne peut pas produire.
- Z3, appele directement (sans le chemin SFAz3 cape), **n'a pas** le mur des 21 caracteres : c'est le resolveur reel du benchmark Sudoku multi-paradigmes.
- Le **fork Automata 2026** leve le **mur 2** (cap du temoin) - temoin `A & ~B` non-cape depuis `&`/`~` - mais **pas le mur 1** (explosion DFA a l'echelle). Voir section 6b et notebook 06.